# 3.2 Test Set Validation: Demographic and SBR Feature Analysis

This notebook validates the findings from the validation set analysis by:
1. Running the same analysis on the **train set** (which serves as our test set)
2. Comparing performance between validation and train sets
3. Assessing model generalization and robustness

## Dataset Split:
- **Validation set** (analyzed in 3.1): Used for initial model development
- **Train set** (analyzed here): Held-out test set for validation

## Objectives:
1. Confirm that imaging features predict SBR pathology on independent data
2. Verify that demographics provide minimal added value across datasets
3. Compare R² scores and assess consistency of findings

## Section 1: Setup and Data Preparation

### 1.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

print("Libraries imported successfully!")

### 1.2 Define Constants

In [ ]:
# Target variables
TARGETS = ['SBR_PC1', 'SBR_PC2', 'SBR_PC3']

# Demographic covariates
DEMO_COLS = ['AGE_AT_VISIT', 'SEX']

# Image feature columns
FEATURE_COLS = [f'latent_{i}' for i in range(256)]

# SBR columns for PCA
SBR_COLS = [
    'DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L',
    'DATSCAN_PUTAMEN_R', 'DATSCAN_PUTAMEN_L',
    'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L_ANT'
]

# Model parameters
ALPHA = 10.0
N_SPLITS = 5
RANDOM_STATE = 42

print(f"Target variables: {TARGETS}")
print(f"Demographic columns: {DEMO_COLS}")
print(f"Number of image features: {len(FEATURE_COLS)}")
print(f"Ridge alpha: {ALPHA}")
print(f"CV folds: {N_SPLITS}")

### 1.3 Load Data

In [ ]:
# Load merged clinical/demographic data
df_merged = pd.read_csv('output/merged_data.csv')
print(f"Merged clinical data shape: {df_merged.shape}")
print(f"Unique patients: {df_merged['PATNO'].nunique()}")

# Load TRAIN set latent vectors (this is our test set)
latent_file = 'output/Experiments/LatentVectorAnalysis/train_latent_vectors_with_patno.csv'
if Path(latent_file).exists():
    df_latent = pd.read_csv(latent_file)
    print(f"\nTrain set latent vectors loaded: {df_latent.shape}")
    print(f"Unique PATNOs: {df_latent['PATNO'].nunique()}")
    print(f"Missing PATNOs: {df_latent['PATNO'].isna().sum()}")
else:
    print(f"\n⚠️ WARNING: {latent_file} not found!")
    print("Please run: python extract_latent_with_patno.py")
    raise FileNotFoundError(f"Required file not found: {latent_file}")

### 1.4 Merge Datasets

In [ ]:
# Remove rows with missing PATNO
df_latent_clean = df_latent.dropna(subset=['PATNO']).copy()
print(f"Latent vectors after removing missing PATNOs: {len(df_latent_clean)}")

# Merge on PATNO
print("\nMerging datasets on PATNO...")
df_combined = pd.merge(
    df_latent_clean,
    df_merged,
    on='PATNO',
    how='inner'
)

print(f"Combined dataset shape: {df_combined.shape}")
print(f"Unique patients: {df_combined['PATNO'].nunique()}")

# Check visits per patient
visits_per_patient = df_combined.groupby('PATNO').size()
print(f"\nVisits per patient: mean={visits_per_patient.mean():.1f}, median={visits_per_patient.median():.0f}")
print(f"Patients with multiple visits: {(visits_per_patient > 1).sum()}")

### 1.5 Calculate SBR Principal Components

In [ ]:
print("Calculating SBR Principal Components...")
print(f"Rows with complete SBR data: {df_combined[SBR_COLS].notna().all(axis=1).sum()}")

# Filter to rows with complete SBR data
df_with_sbr = df_combined.dropna(subset=SBR_COLS).copy()
print(f"Dataset after filtering for complete SBR data: {df_with_sbr.shape}")

# Standardize SBR values
scaler_sbr = StandardScaler()
sbr_scaled = scaler_sbr.fit_transform(df_with_sbr[SBR_COLS])

# Apply PCA
pca = PCA(n_components=3, random_state=0)
sbr_pcs = pca.fit_transform(sbr_scaled)

# Add PC scores
df_with_sbr['SBR_PC1'] = sbr_pcs[:, 0]
df_with_sbr['SBR_PC2'] = sbr_pcs[:, 1]
df_with_sbr['SBR_PC3'] = sbr_pcs[:, 2]

print(f"\nExplained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")

# Update df_combined
df_combined = df_with_sbr.copy()
print(f"\nFinal df_combined shape: {df_combined.shape}")

### 1.6 Select Best Visit per Patient

In [ ]:
def select_best_visit(group):
    """Select the best visit for each patient based on data availability and visit type"""
    # Filter to rows with all required data
    valid_rows = group.dropna(subset=TARGETS + DEMO_COLS)
    
    if len(valid_rows) == 0:
        return None
    
    # Priority order for EVENT_ID
    if 'EVENT_ID' in valid_rows.columns:
        priority = {'SC': 0, 'BL': 1, 'V01': 2, 'V02': 3}
        valid_rows = valid_rows.copy()
        valid_rows['priority'] = valid_rows['EVENT_ID'].map(priority).fillna(99)
        best_row = valid_rows.sort_values('priority').iloc[0:1]
    else:
        best_row = valid_rows.iloc[0:1]
    
    return best_row

print("Selecting best visit for each patient...")
df_final = df_combined.groupby('PATNO', group_keys=False).apply(select_best_visit)
df_final = df_final.dropna(subset=['PATNO']).reset_index(drop=True)

print(f"\nFinal dataset shape: {df_final.shape}")
print(f"Unique patients: {df_final['PATNO'].nunique()}")
print(f"✓ One row per patient: {len(df_final) == df_final['PATNO'].nunique()}")

### 1.7 Extract Features and Targets

In [ ]:
# Extract image features
X_features = df_final[FEATURE_COLS].values
print(f"Image features (X) shape: {X_features.shape}")

# Extract target variables
Y_targets = df_final[TARGETS].values
print(f"Target variables (Y) shape: {Y_targets.shape}")

# Extract demographics
demo_data = df_final[DEMO_COLS].copy()
print(f"Demographics shape: {demo_data.shape}")

# Summary statistics
print(f"\nTarget variable statistics:")
print(df_final[TARGETS].describe())

print(f"\nDemographic statistics:")
print(f"Age - Mean: {demo_data['AGE_AT_VISIT'].mean():.2f}, Std: {demo_data['AGE_AT_VISIT'].std():.2f}")
print(f"Sex distribution:\n{demo_data['SEX'].value_counts()}")

### 1.8 Process Covariates

In [ ]:
# One-Hot Encode SEX
sex_values = demo_data['SEX'].values.reshape(-1, 1)
encoder = OneHotEncoder(sparse_output=False, drop='first')
sex_encoded = encoder.fit_transform(sex_values)
print(f"Sex encoded shape: {sex_encoded.shape}")

# Standardize AGE
age_values = demo_data['AGE_AT_VISIT'].values.reshape(-1, 1)
age_scaler = StandardScaler()
age_scaled = age_scaler.fit_transform(age_values)
print(f"Age scaled shape: {age_scaled.shape}")

# Combine demographics
Z_demo = np.hstack([age_scaled, sex_encoded])
print(f"\nCombined demographic features (Z_demo) shape: {Z_demo.shape}")

### 1.9 Standardize Image Features

In [ ]:
# Standardize image features
feature_scaler = StandardScaler()
X_scaled = feature_scaler.fit_transform(X_features)

print(f"Standardized image features shape: {X_scaled.shape}")
print(f"Mean: {X_scaled.mean():.6f}, Std: {X_scaled.std():.6f}")

## Section 2: Model Training and Evaluation

### 2.1 Initialize Results Storage

In [ ]:
# Dictionary to store results
results = {
    'Target': [],
    'Model': [],
    'R2_Score_Test': [],
    'R2_Score_CV_Mean': [],
    'R2_Score_CV_Std': [],
    'MSE_Test': [],
    'RMSE_Test': []
}

print(f"Ridge alpha: {ALPHA}")
print(f"Cross-validation folds: {N_SPLITS}")
print(f"Random state: {RANDOM_STATE}")
print(f"\nNote: Using CV for robust evaluation with sample size n={len(X_features)}")

### 2.2 Train and Evaluate Models

In [ ]:
# Setup cross-validation
kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for idx, target_name in enumerate(TARGETS):
    print("=" * 80)
    print(f"TARGET: {target_name}")
    print("=" * 80)
    
    # Extract target variable
    y = Y_targets[:, idx]
    
    # -------------------------------------------------------------------------
    # Model Set A: Image Features Only
    # -------------------------------------------------------------------------
    print(f"\n[Model A] Image Features Only")
    
    # Cross-validation
    cv_scores_A = cross_val_score(
        Ridge(alpha=ALPHA, random_state=RANDOM_STATE),
        X_scaled, y, cv=kfold, scoring='r2'
    )
    
    # Train-test split
    X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(
        X_scaled, y, test_size=0.2, random_state=RANDOM_STATE
    )
    
    # Train model
    model_A = Ridge(alpha=ALPHA, random_state=RANDOM_STATE)
    model_A.fit(X_train_A, y_train_A)
    
    # Evaluate
    y_pred_A = model_A.predict(X_test_A)
    r2_test_A = r2_score(y_test_A, y_pred_A)
    mse_A = mean_squared_error(y_test_A, y_pred_A)
    rmse_A = np.sqrt(mse_A)
    
    print(f"  Test R² Score: {r2_test_A:.4f}")
    print(f"  CV R² Score: {cv_scores_A.mean():.4f} (±{cv_scores_A.std():.4f})")
    print(f"  Test MSE: {mse_A:.4f}")
    print(f"  Test RMSE: {rmse_A:.4f}")
    
    # Store results
    results['Target'].append(target_name)
    results['Model'].append('Image Features Only')
    results['R2_Score_Test'].append(r2_test_A)
    results['R2_Score_CV_Mean'].append(cv_scores_A.mean())
    results['R2_Score_CV_Std'].append(cv_scores_A.std())
    results['MSE_Test'].append(mse_A)
    results['RMSE_Test'].append(rmse_A)
    
    # -------------------------------------------------------------------------
    # Model Set B: Image Features + Demographics
    # -------------------------------------------------------------------------
    print(f"\n[Model B] Image Features + Demographics")
    
    # Combine features
    X_combined = np.hstack([X_scaled, Z_demo])
    print(f"  Combined features shape: {X_combined.shape}")
    
    # Cross-validation
    cv_scores_B = cross_val_score(
        Ridge(alpha=ALPHA, random_state=RANDOM_STATE),
        X_combined, y, cv=kfold, scoring='r2'
    )
    
    # Train-test split
    X_train_B, X_test_B, y_train_B, y_test_B = train_test_split(
        X_combined, y, test_size=0.2, random_state=RANDOM_STATE
    )
    
    # Train model
    model_B = Ridge(alpha=ALPHA, random_state=RANDOM_STATE)
    model_B.fit(X_train_B, y_train_B)
    
    # Evaluate
    y_pred_B = model_B.predict(X_test_B)
    r2_test_B = r2_score(y_test_B, y_pred_B)
    mse_B = mean_squared_error(y_test_B, y_pred_B)
    rmse_B = np.sqrt(mse_B)
    
    print(f"  Test R² Score: {r2_test_B:.4f}")
    print(f"  CV R² Score: {cv_scores_B.mean():.4f} (±{cv_scores_B.std():.4f})")
    print(f"  Test MSE: {mse_B:.4f}")
    print(f"  Test RMSE: {rmse_B:.4f}")
    
    # Store results
    results['Target'].append(target_name)
    results['Model'].append('Image Features + Demographics')
    results['R2_Score_Test'].append(r2_test_B)
    results['R2_Score_CV_Mean'].append(cv_scores_B.mean())
    results['R2_Score_CV_Std'].append(cv_scores_B.std())
    results['MSE_Test'].append(mse_B)
    results['RMSE_Test'].append(rmse_B)
    
    # Calculate improvement
    r2_improvement = r2_test_B - r2_test_A
    cv_improvement = cv_scores_B.mean() - cv_scores_A.mean()
    mse_reduction = ((mse_A - mse_B) / mse_A) * 100 if mse_A != 0 else 0
    
    print(f"\n[Improvement]")
    print(f"  ΔR² (Test): {r2_improvement:+.4f}")
    print(f"  ΔR² (CV): {cv_improvement:+.4f}")
    print(f"  MSE Reduction: {mse_reduction:.2f}%")
    print()

print("\n" + "=" * 80)
print("MODEL TRAINING COMPLETE")
print("=" * 80)

## Section 3: Results and Comparison

### 3.1 Test Set Results Summary

In [ ]:
# Create results DataFrame
df_results_test = pd.DataFrame(results)

# Pivot tables
df_pivot_r2_test = df_results_test.pivot(index='Target', columns='Model', values='R2_Score_Test')
df_pivot_r2_cv = df_results_test.pivot(index='Target', columns='Model', values='R2_Score_CV_Mean')
df_pivot_mse = df_results_test.pivot(index='Target', columns='Model', values='MSE_Test')

print("\n" + "=" * 80)
print("TEST SET RESULTS SUMMARY")
print("=" * 80)

print("\nTest Set R² Scores:")
print(df_pivot_r2_test.to_string())

print("\n\nCross-Validation R² Scores (Mean ± Std):")
for target in TARGETS:
    print(f"\n{target}:")
    for model in ['Image Features Only', 'Image Features + Demographics']:
        mean_val = df_results_test[(df_results_test['Target']==target) & (df_results_test['Model']==model)]['R2_Score_CV_Mean'].values[0]
        std_val = df_results_test[(df_results_test['Target']==target) & (df_results_test['Model']==model)]['R2_Score_CV_Std'].values[0]
        print(f"  {model}: {mean_val:.4f} (±{std_val:.4f})")

print("\n\nMean Squared Error (MSE):")
print(df_pivot_mse.to_string())

### 3.2 Load Validation Set Results for Comparison

In [ ]:
# Load validation set results from previous analysis
val_results_path = 'output/demographic_sbr_feature_analysis_results.csv'

if Path(val_results_path).exists():
    df_results_val = pd.read_csv(val_results_path)
    print(f"Loaded validation set results: {df_results_val.shape}")
    
    # Pivot validation results
    df_pivot_r2_val_test = df_results_val.pivot(index='Target', columns='Model', values='R2_Score_Test')
    df_pivot_r2_val_cv = df_results_val.pivot(index='Target', columns='Model', values='R2_Score_CV_Mean')
    
    print("\nValidation Set R² Scores (Test):")
    print(df_pivot_r2_val_test.to_string())
    
    print("\nValidation Set R² Scores (CV):")
    print(df_pivot_r2_val_cv.to_string())
else:
    print(f"⚠️ Validation results not found at {val_results_path}")
    print("Run notebook 3.1 first to generate validation results.")
    df_results_val = None

### 3.3 Validation vs Test Set Comparison

In [ ]:
if df_results_val is not None:
    print("\n" + "=" * 80)
    print("VALIDATION vs TEST SET COMPARISON")
    print("=" * 80)
    
    comparison_data = []
    
    for target in TARGETS:
        for model in ['Image Features Only', 'Image Features + Demographics']:
            # Validation set scores
            val_test_r2 = df_results_val[(df_results_val['Target']==target) & (df_results_val['Model']==model)]['R2_Score_Test'].values[0]
            val_cv_r2 = df_results_val[(df_results_val['Target']==target) & (df_results_val['Model']==model)]['R2_Score_CV_Mean'].values[0]
            
            # Test set scores
            test_test_r2 = df_results_test[(df_results_test['Target']==target) & (df_results_test['Model']==model)]['R2_Score_Test'].values[0]
            test_cv_r2 = df_results_test[(df_results_test['Target']==target) & (df_results_test['Model']==model)]['R2_Score_CV_Mean'].values[0]
            
            # Calculate differences
            diff_test_r2 = test_test_r2 - val_test_r2
            diff_cv_r2 = test_cv_r2 - val_cv_r2
            
            comparison_data.append({
                'Target': target,
                'Model': model,
                'Val_Test_R2': val_test_r2,
                'Test_Test_R2': test_test_r2,
                'Diff_Test_R2': diff_test_r2,
                'Val_CV_R2': val_cv_r2,
                'Test_CV_R2': test_cv_r2,
                'Diff_CV_R2': diff_cv_r2
            })
    
    df_comparison = pd.DataFrame(comparison_data)
    
    print("\nR² Score Comparison (Image Features Only):")
    print(df_comparison[df_comparison['Model']=='Image Features Only'][['Target', 'Val_Test_R2', 'Test_Test_R2', 'Diff_Test_R2', 'Val_CV_R2', 'Test_CV_R2', 'Diff_CV_R2']].to_string(index=False))
    
    print("\n\nR² Score Comparison (Image Features + Demographics):")
    print(df_comparison[df_comparison['Model']=='Image Features + Demographics'][['Target', 'Val_Test_R2', 'Test_Test_R2', 'Diff_Test_R2', 'Val_CV_R2', 'Test_CV_R2', 'Diff_CV_R2']].to_string(index=False))
    
    # Summary statistics
    print("\n\nSummary Statistics:")
    print(f"Mean absolute difference (Test R²): {df_comparison['Diff_Test_R2'].abs().mean():.4f}")
    print(f"Mean absolute difference (CV R²): {df_comparison['Diff_CV_R2'].abs().mean():.4f}")
    print(f"Max absolute difference (Test R²): {df_comparison['Diff_Test_R2'].abs().max():.4f}")
    print(f"Max absolute difference (CV R²): {df_comparison['Diff_CV_R2'].abs().max():.4f}")

### 3.4 Visualization: Validation vs Test Comparison

In [ ]:
if df_results_val is not None:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Test R² Comparison (Image Only)
    img_only = df_comparison[df_comparison['Model']=='Image Features Only']
    x = np.arange(len(TARGETS))
    width = 0.35
    
    axes[0, 0].bar(x - width/2, img_only['Val_Test_R2'], width, label='Validation Set', color='steelblue', alpha=0.8, edgecolor='black')
    axes[0, 0].bar(x + width/2, img_only['Test_Test_R2'], width, label='Test Set', color='coral', alpha=0.8, edgecolor='black')
    axes[0, 0].set_title('Test R² Comparison: Image Features Only', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Target Variable', fontsize=12)
    axes[0, 0].set_ylabel('R² Score', fontsize=12)
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(TARGETS)
    axes[0, 0].legend(fontsize=10)
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # Plot 2: CV R² Comparison (Image Only)
    axes[0, 1].bar(x - width/2, img_only['Val_CV_R2'], width, label='Validation Set', color='steelblue', alpha=0.8, edgecolor='black')
    axes[0, 1].bar(x + width/2, img_only['Test_CV_R2'], width, label='Test Set', color='coral', alpha=0.8, edgecolor='black')
    axes[0, 1].set_title('CV R² Comparison: Image Features Only', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Target Variable', fontsize=12)
    axes[0, 1].set_ylabel('R² Score (CV)', fontsize=12)
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(TARGETS)
    axes[0, 1].legend(fontsize=10)
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Test R² Comparison (Image + Demo)
    img_demo = df_comparison[df_comparison['Model']=='Image Features + Demographics']
    
    axes[1, 0].bar(x - width/2, img_demo['Val_Test_R2'], width, label='Validation Set', color='mediumseagreen', alpha=0.8, edgecolor='black')
    axes[1, 0].bar(x + width/2, img_demo['Test_Test_R2'], width, label='Test Set', color='indianred', alpha=0.8, edgecolor='black')
    axes[1, 0].set_title('Test R² Comparison: Image + Demographics', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Target Variable', fontsize=12)
    axes[1, 0].set_ylabel('R² Score', fontsize=12)
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(TARGETS)
    axes[1, 0].legend(fontsize=10)
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Plot 4: Difference in R² (Test - Validation)
    axes[1, 1].bar(x - width/2, img_only['Diff_CV_R2'], width, label='Image Only', color='steelblue', alpha=0.8, edgecolor='black')
    axes[1, 1].bar(x + width/2, img_demo['Diff_CV_R2'], width, label='Image + Demo', color='mediumseagreen', alpha=0.8, edgecolor='black')
    axes[1, 1].set_title('R² Difference (Test - Validation) [CV Scores]', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Target Variable', fontsize=12)
    axes[1, 1].set_ylabel('ΔR² (Test - Val)', fontsize=12)
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(TARGETS)
    axes[1, 1].legend(fontsize=10)
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    
    plt.tight_layout()
    plt.savefig('output/validation_vs_test_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nPlot saved to: output/validation_vs_test_comparison.png")

### 3.5 Save Results

In [ ]:
# Save test set results
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

df_results_test.to_csv(output_dir / 'test_set_results.csv', index=False)
df_pivot_r2_test.to_csv(output_dir / 'test_set_r2_scores_test.csv')
df_pivot_r2_cv.to_csv(output_dir / 'test_set_r2_scores_cv.csv')
df_pivot_mse.to_csv(output_dir / 'test_set_mse_scores.csv')

if df_results_val is not None:
    df_comparison.to_csv(output_dir / 'validation_vs_test_comparison.csv', index=False)
    print(f"Comparison results saved to: {output_dir / 'validation_vs_test_comparison.csv'}")

print("\nTest set results saved to:")
print(f"  - {output_dir / 'test_set_results.csv'}")
print(f"  - {output_dir / 'test_set_r2_scores_test.csv'}")
print(f"  - {output_dir / 'test_set_r2_scores_cv.csv'}")
print(f"  - {output_dir / 'test_set_mse_scores.csv'}")

## Summary

This notebook validated the findings from the validation set analysis by:
1. ✅ Running the same analysis on the train set (held-out test data)
2. ✅ Comparing performance between validation and test sets
3. ✅ Assessing model generalization and consistency

### Key Findings:
- **Consistency**: Compare R² scores across datasets to assess generalization
- **Robustness**: Small differences indicate robust model performance
- **Demographics**: Confirm whether demographics provide value across both datasets

### Interpretation Guidelines:
- **Small differences (< 0.05)**: Excellent generalization
- **Moderate differences (0.05-0.10)**: Acceptable, some dataset-specific variation
- **Large differences (> 0.10)**: Potential overfitting or dataset bias

### Next Steps:
1. Feature importance analysis to understand key predictive dimensions
2. Clinical subgroup analysis (PD vs Control vs SWEDD)
3. Longitudinal analysis if multiple timepoints available